# 02 – Feature exploration

Explores the candidate feature columns from the **Dec 2018 snapshot**, for the trials in `analysis.cohort`. The goal is to decide which columns are usable as features (see `docs/feature_list.md`).

## Section 1 – Load the one-per-trial columns

The cohort joined to `studies`, `designs` and `eligibilities` (each has one row per trial).

In [1]:
import pandas as pd
from sqlalchemy import text
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))  

from src.db import engine

query = text("""
    SELECT 
        -- Cohort keys and split
        c.nct_id,
        c.label,
        c.split,
        c.status_2018,
        
        -- Studies candidates
        s.phase,
        s.enrollment,
        s.enrollment_type,
        s.start_date,
        s.primary_completion_date,
        s.number_of_arms,
        s.has_dmc,
        
        -- Designs candidates
        d.allocation,
        d.masking,
        d.primary_purpose,
        d.intervention_model,
        
        -- Eligibilities candidates
        e.gender,
        e.minimum_age,
        e.maximum_age,
        e.healthy_volunteers

    FROM analysis.cohort c
    LEFT JOIN snap_2018_12.studies s ON c.nct_id = s.nct_id
    LEFT JOIN snap_2018_12.designs d ON c.nct_id = d.nct_id
    LEFT JOIN snap_2018_12.eligibilities e ON c.nct_id = e.nct_id;
""")

with engine.connect() as conn:
    df_trials = pd.read_sql(query, conn)



### Check: row count (expect 47,210, so the joins didn't multiply rows)

In [2]:
print(df_trials.shape)

(47210, 19)


## Section 2 – Explore the one-per-trial columns

### Missing values (% of rows)

In [3]:
display(((df_trials.isna().mean())*100).round(2))


nct_id                      0.00
label                      32.42
split                       0.00
status_2018                 0.00
phase                       0.00
enrollment                  0.04
enrollment_type             0.05
start_date                  0.03
primary_completion_date     0.01
number_of_arms              0.71
has_dmc                    11.87
allocation                 27.40
masking                     0.17
primary_purpose             0.31
intervention_model          0.09
gender                      0.00
minimum_age                 0.00
maximum_age                 0.00
healthy_volunteers          0.00
dtype: float64

### Categorical columns: value counts (including missing)

In [4]:
cat_cols = [
    'phase', 
    'enrollment_type', 
    'allocation', 
    'masking', 
    'primary_purpose', 
    'intervention_model', 
    'gender', 
    'healthy_volunteers'
]
for col in cat_cols:
    display(df_trials[col].value_counts(dropna=False))

phase
N/A                24060
Phase 2             7267
Phase 3             3995
Phase 1             3912
Phase 4             3787
Phase 1/Phase 2     2401
Phase 2/Phase 3      954
Early Phase 1        834
Name: count, dtype: int64

enrollment_type
Anticipated    47187
NaN               23
Name: count, dtype: int64

allocation
Randomized        29443
NaN               12936
Non-Randomized     4831
Name: count, dtype: int64

masking
None (Open Label)    28369
Single                6548
Double                5007
Quadruple             4066
Triple                3140
NaN                     80
Name: count, dtype: int64

primary_purpose
Treatment                   30787
Prevention                   4566
Other                        2820
Diagnostic                   2697
Supportive Care              2311
Basic Science                2144
Health Services Research     1109
Screening                     425
Device Feasibility            203
NaN                           148
Name: count, dtype: int64

intervention_model
Parallel Assignment        28717
Single Group Assignment    14021
Crossover Assignment        2665
Sequential Assignment       1110
Factorial Assignment         654
NaN                           43
Name: count, dtype: int64

gender
All       40617
Female     4836
Male       1757
Name: count, dtype: int64

healthy_volunteers
No                            37307
Accepts Healthy Volunteers     9888
                                 15
Name: count, dtype: int64

### Numeric columns: summary statistics

In [5]:
display(df_trials[['enrollment','number_of_arms']].describe())


,enrollment,number_of_arms
count,4.719000e+04,46877.000000
mean,4.698630e+03,2.051710
std,6.511029e+05,1.176256
min,1.000000e+00,1.000000
25%,4.000000e+01,1.000000
50%,8.000000e+01,2.000000
75%,1.950000e+02,2.000000
max,1.000000e+08,31.000000


### Age columns: what the text looks like

In [6]:
display(df_trials[['minimum_age','maximum_age']].head(10))

,minimum_age,maximum_age
0,18 Years,N/A
1,35 Years,42 Years
2,18 Years,N/A
3,18 Years,75 Years
4,35 Years,80 Years
5,18 Years,65 Years
6,18 Years,N/A
7,18 Years,70 Years
8,20 Years,80 Years
9,18 Years,N/A


### Is the missing `allocation` explained by single-group trials?

In [7]:
pd.crosstab(df_trials['allocation'].fillna('no allocation'),df_trials['intervention_model']).T

allocation,Non-Randomized,Randomized,no allocation
intervention_model,,,
Crossover Assignment,175,2483,7
Factorial Assignment,60,591,3
Parallel Assignment,3227,25451,39
Sequential Assignment,509,375,226
Single Group Assignment,850,538,12633


### The largest `enrollment` values (outlier check)

In [8]:
df_trials.sort_values(by='enrollment',ascending=False).head(10)

,nct_id,label,split,status_2018,phase,enrollment,enrollment_type,start_date,primary_completion_date,number_of_arms,has_dmc,allocation,masking,primary_purpose,intervention_model,gender,minimum_age,maximum_age,healthy_volunteers
42277,NCT01912313,NaN,train,RECRUITING,N/A,99999999.0,Anticipated,2013-08-31,2019-08-31,1.0,False,NaN,None (Open Label),Basic Science,Single Group Assignment,All,18 Years,99 Years,Accepts Healthy Volunteers
38722,NCT02421705,NaN,train,RECRUITING,N/A,99999999.0,Anticipated,2010-02-28,2099-01-31,1.0,False,NaN,None (Open Label),Basic Science,Single Group Assignment,All,18 Years,65 Years,Accepts Healthy Volunteers
25002,NCT03090321,NaN,train,RECRUITING,N/A,2000000.0,Anticipated,2015-03-31,2017-12-31,5.0,False,Randomized,Single,Prevention,Parallel Assignment,All,18 Years,N/A,Accepts Healthy Volunteers
11300,NCT03455478,NaN,test,RECRUITING,N/A,1000000.0,Anticipated,2016-01-01,2018-11-30,2.0,None,Non-Randomized,None (Open Label),Prevention,Parallel Assignment,All,60 Years,100 Years,Accepts Healthy Volunteers
760,NCT03676764,0.0,test,NOT_YET_RECRUITING,Phase 4,447780.0,Anticipated,2019-01-31,2022-01-31,4.0,True,Randomized,Quadruple,Prevention,Factorial Assignment,All,1 Month,59 Months,Accepts Healthy Volunteers
42429,NCT01864603,0.0,train,RECRUITING,N/A,340000.0,Anticipated,2013-04-30,2020-07-31,2.0,True,Randomized,None (Open Label),Other,Parallel Assignment,All,N/A,N/A,Accepts Healthy Volunteers
16130,NCT03335072,NaN,valid,NOT_YET_RECRUITING,Phase 4,320000.0,Anticipated,2019-06-01,2022-07-01,4.0,True,Randomized,None (Open Label),Treatment,Parallel Assignment,All,N/A,N/A,No
27424,NCT03014167,0.0,train,RECRUITING,Phase 4,240000.0,Anticipated,2017-10-04,2022-12-31,2.0,True,Randomized,None (Open Label),Treatment,Single Group Assignment,All,12 Months,N/A,Accepts Healthy Volunteers
38996,NCT02392832,0.0,train,RECRUITING,N/A,240000.0,Anticipated,2015-04-30,2018-09-30,2.0,False,Randomized,Single,Other,Crossover Assignment,All,7 Months,N/A,No
16062,NCT03338244,0.0,valid,RECRUITING,Phase 4,227000.0,Anticipated,2017-10-20,2019-07-31,2.0,True,Randomized,None (Open Label),Treatment,Parallel Assignment,All,1 Month,60 Months,No


### Units used in `maximum_age`

In [9]:
df_trials['maximum_age'].str.extract(r'^\d+\s*([a-zA-Z]+)')[0].value_counts(dropna=False)

0
Years      24212
NaN        22098
Months       410
Weeks        182
Days         173
Hours         72
Year          22
Minutes       16
Month         11
Day           10
Hour           2
Week           1
Minute         1
Name: count, dtype: int64

### Units used in `minimum_age`

In [10]:
df_trials['minimum_age'].str.extract(r'^\d+\s*([a-zA-Z]+)')[0].value_counts(dropna=False)

0
Years     43510
NaN        2468
Months      638
Year        336
Weeks       129
Month       107
Days         22
Name: count, dtype: int64

### Findings – Section 2

- **Row count is 47,210** after joining `studies`, `designs` and `eligibilities`, so these joins don't multiply rows.
- **`enrollment_type` is "Anticipated" for every trial** (47,187; 23 missing). This confirms that the 2018 enrolment is the *target* size, not the actual count, so it's safe from leakage. The column itself never varies, so it's **dropped** as a feature.
- **`allocation` is 27.4% missing, and that's not random.** 12,633 of the 12,936 missing values (97.7%) are single-group trials, which have nobody to randomise. Fill with a category such as "single group".
- **`phase` = "N/A" for 51%.** These are non-drug trials (devices, behavioural…), so "N/A" is a real category, not missing data.
- **`enrollment` has extreme values.** The median is 80, but the max is 99,999,999 (a placeholder, 2 trials), and there are several values in the 200,000–2,000,000 range. Treat the placeholder as missing, and use a log scale.
- **Placeholder dates exist.** For example, `primary_completion_date` = 2099-01-31. Check for these before computing planned duration.
- **The ages are text** ("18 Years"), with units from Minutes to Years and both singular and plural forms. "N/A" means *no limit*: 46.8% have no maximum age and 5.2% have no minimum. These need parsing to a number in years, plus a "no limit" flag.
- **`healthy_volunteers` hides 15 missing values as empty strings** (`""`), which `isna()` doesn't count.
- **`has_dmc` is 11.9% missing.** Keep "missing" as its own category.

## Section 3 – Explore the many-per-trial tables

These tables have **several rows per trial**, so each one must later be squashed to one row per trial before joining. Every query starts from `analysis.cohort` with a `LEFT JOIN`, so trials with no rows in the table are kept (with empty values).

### Facilities: load the site rows for cohort trials

In [11]:
query = text("""
    SELECT
        c.nct_id,
        f.country
    FROM analysis.cohort c
    LEFT JOIN snap_2018_12.facilities f ON c.nct_id=f.nct_id

""")
with engine.connect() as conn:
    facilities = pd.read_sql(query, conn)

### Facilities: first rows

In [12]:
display(facilities.head())

,nct_id,country
0,NCT03703765,France
1,NCT03703700,China
2,NCT03703700,China
3,NCT03703700,China
4,NCT03703700,China


### Facilities: sites per trial, and trials per country

In [13]:
# .count() skips NaN, so a trial with no facility rows gets 0 sites (not 1)
sites_per_trial = (facilities.groupby("nct_id").count()
                   .rename(columns={"country": "n_sites"})
                   .sort_values("n_sites", ascending=False))

# .nunique() counts distinct trials per country, not site rows
trials_per_country = (facilities.groupby("country").nunique()
                      .rename(columns={"nct_id": "n_trials"})
                      .sort_values("n_trials", ascending=False))

display(sites_per_trial.describe())
display(trials_per_country.head(10))

,n_sites
count,47210.000000
mean,6.338572
std,37.889353
min,0.000000
25%,1.000000
50%,1.000000
75%,2.000000
max,1457.000000


,n_trials
country,
United States,18544
France,4035
China,3995
Canada,3125
United Kingdom,2288
Germany,2098
Spain,1871
Italy,1724
"Korea, Republic of",1608


In [14]:
from src.db import load_cohort_table

### Countries: load (one row per trial per country, plus a `removed` flag)

In [15]:
countries = load_cohort_table("countries", ["name", "removed"])
display(countries.head(10))

,nct_id,name,removed
0,NCT03703765,France,None
1,NCT03703700,China,None
2,NCT03703687,NaN,None
3,NCT03703674,India,None
4,NCT03703635,China,None
5,NCT03703622,Uganda,None
6,NCT03703609,NaN,None
7,NCT03703596,China,None
8,NCT03703570,Japan,None
9,NCT03703557,France,None


### Countries: what does `removed` contain?

`True` means the country was listed earlier and later removed from the trial's record; empty (`None`) means it's still listed.

In [16]:
display(countries["removed"].value_counts(dropna=False))

removed
None    66713
True      809
Name: count, dtype: int64

### Countries: countries per trial (removed countries not counted)

Two traps here:
- `removed` holds Python `True`/`None`, not the text `"True"`, so comparing with the string `"True"` matched nothing and nothing was filtered out.
- Dropping the removed rows would make a trial whose *only* country was removed disappear from the count. Blanking the name instead keeps the trial, with 0.

In [17]:
# True for removed rows; None (not removed) and NaN (no country at all) count as False
is_removed = countries["removed"].eq(True)

# Blank out the name of removed rows instead of dropping them, so every trial stays
countries_active = countries.assign(name=countries["name"].where(~is_removed))

countries_per_trial = (countries_active.groupby("nct_id")["name"].count()
                       .rename("n_countries"))
trials_with_removed = countries.loc[is_removed, "nct_id"].nunique()

display(countries_per_trial.describe())
print(f"Trials with at least one removed country: {trials_with_removed:,} "
      f"({trials_with_removed / countries['nct_id'].nunique():.1%} of the cohort)")

count    47210.000000
mean         1.303304
std          2.419200
min          0.000000
25%          1.000000
50%          1.000000
75%          1.000000
max         49.000000
Name: n_countries, dtype: float64

Trials with at least one removed country: 519 (1.1% of the cohort)


### Countries: does the `countries` table agree with `facilities`?

The same quantity computed two ways: distinct site countries per trial vs. the `countries` table.

In [18]:
# Distinct countries per trial from the site rows (.nunique() skips NaN, so no sites -> 0)
countries_from_facilities = (facilities.groupby("nct_id")["country"].nunique()
                             .rename("n_countries_facilities"))

country_check = pd.concat([countries_per_trial, countries_from_facilities], axis=1)
country_check["match"] = country_check["n_countries"] == country_check["n_countries_facilities"]

display(country_check["match"].value_counts())
display(country_check[~country_check["match"]].head(10))

match
True    47210
Name: count, dtype: int64

,n_countries,n_countries_facilities,match
nct_id,,,


### Countries: top 10 by number of trials

In [19]:
# One row per trial per country, so counting names counts trials
display(countries_active["name"].value_counts().head(10))

name
United States         18544
France                 4035
China                  3995
Canada                 3125
United Kingdom         2288
Germany                2098
Spain                  1871
Italy                  1724
Korea, Republic of     1608
Belgium                1236
Name: count, dtype: int64

### Sponsors: load

In [20]:
sponsors = load_cohort_table("sponsors", ["agency_class", "lead_or_collaborator"])
display(sponsors.head())
display(sponsors["lead_or_collaborator"].value_counts(dropna=False))

,nct_id,agency_class,lead_or_collaborator
0,NCT03703765,Other,lead
1,NCT03703700,Other,lead
2,NCT03703687,Other,lead
3,NCT03703674,Other,lead
4,NCT03703635,Other,lead


lead_or_collaborator
lead            47210
collaborator    34024
Name: count, dtype: int64

### Sponsors: does every trial have exactly one lead sponsor?

If every trial shows 0 leads, check the spelling of `lead` in the value counts above.

In [21]:
is_lead = sponsors["lead_or_collaborator"].eq("lead")

# Summing a True/False column per trial counts the True values
leads_per_trial = is_lead.groupby(sponsors["nct_id"]).sum().rename("n_leads")
display(leads_per_trial.value_counts().sort_index())

n_leads
1    47210
Name: count, dtype: int64

### Sponsors: lead sponsor type, and number of collaborators

In [22]:
# Lead sponsor's class per trial; reindex so trials without a lead show up as NaN
lead_agency_class = (sponsors[is_lead].groupby("nct_id")["agency_class"].first()
                     .reindex(leads_per_trial.index)
                     .rename("lead_agency_class"))

is_collaborator = sponsors["lead_or_collaborator"].eq("collaborator")
collaborators_per_trial = (is_collaborator.groupby(sponsors["nct_id"]).sum()
                           .rename("n_collaborators"))

display(lead_agency_class.value_counts(dropna=False))
display(collaborators_per_trial.describe())

lead_agency_class
Other       38277
Industry     7775
NIH           621
U.S. Fed      537
Name: count, dtype: int64

count    47210.000000
mean         0.720695
std          1.730583
min          0.000000
25%          0.000000
50%          0.000000
75%          1.000000
max         68.000000
Name: n_collaborators, dtype: float64

### Interventions: load

In [23]:
interventions = load_cohort_table("interventions", ["intervention_type"])
display(interventions.head())

,nct_id,intervention_type
0,NCT03703765,Device
1,NCT03703700,Drug
2,NCT03703700,Drug
3,NCT03703687,Behavioral
4,NCT03703674,Drug


### Interventions: interventions per trial, and share of trials using each type

A trial can combine several types (e.g. a drug plus a behavioural intervention), so the shares add up to more than 100%.

In [24]:
interventions_per_trial = (interventions.groupby("nct_id")["intervention_type"].count()
                           .rename("n_interventions"))

# Keep one row per (trial, type), then count trials per type
trials_per_type = (interventions.dropna(subset=["intervention_type"])
                   .drop_duplicates(["nct_id", "intervention_type"])["intervention_type"]
                   .value_counts())
n_trials = interventions["nct_id"].nunique()

display(interventions_per_trial.describe())
display((trials_per_type / n_trials).round(3).rename("share_of_trials"))

count    47210.000000
mean         2.033785
std          1.452614
min          1.000000
25%          1.000000
50%          2.000000
75%          2.000000
max         35.000000
Name: n_interventions, dtype: float64

intervention_type
Drug                   0.439
Other                  0.204
Device                 0.158
Behavioral             0.124
Procedure              0.120
Biological             0.071
Radiation              0.038
Dietary Supplement     0.031
Diagnostic Test        0.018
Combination Product    0.006
Genetic                0.004
Name: share_of_trials, dtype: float64

### Conditions: load

In [25]:
conditions = load_cohort_table("conditions", ["name"])
display(conditions.head())

,nct_id,name
0,NCT03703765,Venous Disease
1,NCT03703700,"Infertility, Female"
2,NCT03703687,Palliative Care
3,NCT03703674,Alcoholic Hepatitis
4,NCT03703635,Intracranial Artery Stenosis


### Conditions: conditions per trial, and the 30 most common names

In [26]:
conditions_per_trial = (conditions.groupby("nct_id")["name"].count()
                        .rename("n_conditions"))

display(conditions_per_trial.describe())
# Lower-case first, so "Breast Cancer" and "breast cancer" are counted together
display(conditions["name"].str.lower().value_counts().head(30))

count    47210.000000
mean         1.805190
std          2.137242
min          1.000000
25%          1.000000
50%          1.000000
75%          2.000000
max         90.000000
Name: n_conditions, dtype: float64

name
breast cancer                 803
obesity                       588
prostate cancer               478
stroke                        477
depression                    429
healthy                       402
hypertension                  343
heart failure                 315
cancer                        308
colorectal cancer             291
pain                          280
coronary artery disease       267
multiple myeloma              263
non-small cell lung cancer    253
schizophrenia                 251
lung cancer                   249
acute myeloid leukemia        243
atrial fibrillation           233
parkinson disease             230
hepatocellular carcinoma      218
asthma                        214
anxiety                       212
pancreatic cancer             211
melanoma                      208
ovarian cancer                204
diabetes mellitus, type 2     203
physical activity             197
gastric cancer                196
multiple sclerosis            195
lymphoma 

### Conditions: how many trials mention a cancer keyword?

A first draft of the future *oncology* feature. The sample below is for checking that the keywords don't catch the wrong conditions.

In [27]:
cancer_pattern = (r"cancer|carcinoma|tumou?r|neoplasm|lymphoma|leuka?emia|melanoma"
                  r"|sarcoma|myeloma|glioma|glioblastoma|oncolog")

mentions_cancer = conditions["name"].str.contains(cancer_pattern, case=False, na=False)
trial_has_cancer = mentions_cancer.groupby(conditions["nct_id"]).any()

print(f"Trials with a cancer keyword: {trial_has_cancer.sum():,} ({trial_has_cancer.mean():.1%})")
display(conditions.loc[mentions_cancer, "name"].sample(15, random_state=0))

Trials with a cancer keyword: 12,284 (26.0%)


45579                  Relapsed Acute Myelogenous Leukemia
13852                                    Pancreatic Cancer
45517                                   Nasopharynx Cancer
23384                           Non Small Cell Lung Cancer
391                              Recurrent Ependymal Tumor
182                         Refractory Plasma Cell Myeloma
50704                               Urinary Bladder Cancer
11817                             Hepatocellular Carcinoma
79159                Unresectable Hepatocellular Carcinoma
76562                                   Malignant Neoplasm
67870                              Thyroid Gland Carcinoma
61496          Pancreatic Neuroendocrine Tumour Metastatic
21843    Stage IVA Lip and Oral Cavity Squamous Cell Ca...
30616                       Stage IA Breast Cancer AJCC v7
65605                                             Sarcomas
Name: name, dtype: str

### Findings – Section 3

- **Sites (`facilities`):** the median trial has **1 site**, the mean is 6.3, and the max is 1,457. A few huge trials inflate the mean. Some trials list **0 sites**.
- **Countries:** the median is 1 and the max 49. **519 trials (1.1%) had removed a country by Dec 2018**, which was known at the time and is a possible "trouble" signal. Once removed countries are excluded, the `countries` table **matches the countries in `facilities` for all 47,210 trials**, so either source works.
- **Top countries by number of trials:** US (18,544), France, China, Canada, UK… Counting site rows instead would have overstated countries whose trials use many hospitals.
- **Sponsors:** **every trial has exactly one lead sponsor**, so "lead sponsor type" is safe to use. Lead types: Other 81.1%, Industry 16.5%, NIH 1.3%, U.S. Fed 1.1%. Collaborators: median 0, max 68.
- **Interventions:** every trial has at least 1 (median 2). A trial can combine types. Drug is in 43.9% of trials, Other 20.4%, Device 15.8%, Behavioral 12.4%, Procedure 12.0%, Biological 7.1%.
- **Conditions:** every trial has at least 1 (median 1, max 90). Names are messy free text; "healthy" (402) is not a disease.
- **Cancer keywords** match **26.0% of trials** (12,284). The random sample shows only genuine cancers, so the keyword list looks reliable for an *oncology* flag.

## Section 4 – `calculated_values`

AACT pre-computes some columns per trial. Some describe the trial **as registered** (usable), others describe **what happened later** (results, actual duration). Those would leak the outcome, so each column needs a verdict.

### Load all columns for the cohort trials

In [28]:
query = text("""
    SELECT cv.*
    FROM snap_2018_12.calculated_values cv
    WHERE cv.nct_id IN (SELECT nct_id FROM analysis.cohort)
""")
with engine.connect() as conn:
    calculated = pd.read_sql(query, conn)

print(calculated.shape)

(47210, 16)


### Every column: type, missing %, and an example value

In [29]:
column_overview = pd.DataFrame({
    "dtype": calculated.dtypes.astype(str),
    "missing_pct": (calculated.isna().mean() * 100).round(1),
    "example": calculated.iloc[0],
})
display(column_overview)

,dtype,missing_pct,example
id,int64,0.0,19875825
nct_id,str,0.0,NCT03217136
number_of_facilities,float64,11.0,36.0
number_of_nsae_subjects,float64,100.0,NaN
number_of_sae_subjects,float64,100.0,NaN
registered_in_calendar_year,int64,0.0,2017
nlm_download_date,object,0.0,2018-11-30
actual_duration,float64,99.1,NaN
were_results_reported,bool,0.0,False
months_to_report_results,float64,100.0,NaN


### Three example trials, shown vertically

In [30]:
display(calculated.head(3).T)

,0,1,2
id,19875825,19844021,19936658
nct_id,NCT03217136,NCT03627910,NCT02424461
number_of_facilities,36.0,1.0,1.0
number_of_nsae_subjects,NaN,NaN,NaN
number_of_sae_subjects,NaN,NaN,NaN
registered_in_calendar_year,2017,2018,2015
nlm_download_date,2018-11-30,2018-11-30,2018-11-30
actual_duration,NaN,NaN,NaN
were_results_reported,False,False,False
months_to_report_results,NaN,NaN,NaN


### Check: does `number_of_facilities` match the site count from Section 3?

In [31]:
if "number_of_facilities" in calculated.columns:
    facility_check = pd.concat(
        [sites_per_trial["n_sites"],
         calculated.set_index("nct_id")["number_of_facilities"]],
        axis=1,
    )
    # a trial missing from calculated_values has no count: treat it as 0 sites
    facility_check["number_of_facilities"] = facility_check["number_of_facilities"].fillna(0)
    matches = facility_check["n_sites"] == facility_check["number_of_facilities"]
    print(f"Trials where the two counts match: {matches.sum():,} of {len(facility_check):,}")
    display(facility_check[~matches].head(10))

Trials where the two counts match: 47,210 of 47,210


,n_sites,number_of_facilities
nct_id,,


### Check: does `has_us_facility` match the facilities table?

In [32]:
if "has_us_facility" in calculated.columns:
    us_from_facilities = (facilities["country"].eq("United States")
                          .groupby(facilities["nct_id"]).any()
                          .rename("us_from_facilities"))
    us_check = pd.concat(
        [us_from_facilities, calculated.set_index("nct_id")["has_us_facility"]], axis=1
    )
    display(pd.crosstab(us_check["us_from_facilities"],
                        us_check["has_us_facility"].astype(str)))

has_us_facility,False,True
us_from_facilities,,
False,23456,9
True,0,18544


### The numeric age columns (already parsed by AACT)

In [33]:
age_cols = [c for c in ["minimum_age_num", "minimum_age_unit",
                         "maximum_age_num", "maximum_age_unit"] if c in calculated.columns]
display(calculated[age_cols].head(10))
if "minimum_age_unit" in calculated.columns:
    display(calculated["minimum_age_unit"].value_counts(dropna=False))

,minimum_age_num,minimum_age_unit,maximum_age_num,maximum_age_unit
0,NaN,NaN,17.0,Years
1,50.0,Years,80.0,Years
2,18.0,Years,NaN,NaN
3,18.0,Years,NaN,NaN
4,6.0,Years,10.0,Years
5,18.0,Years,45.0,Years
6,18.0,Years,NaN,NaN
7,18.0,Years,NaN,NaN
8,18.0,Years,NaN,NaN
9,18.0,Years,NaN,NaN


minimum_age_unit
 Years     43510
NaN         2468
 Months      638
 Year        336
 Weeks       129
 Month       107
 Days         22
Name: count, dtype: int64

### Verdict per column (known in Dec 2018?)

| Column | Missing | Verdict | Why |
|---|---|---|---|
| `id`, `nct_id` | 0% | — | Identifiers, not features |
| `number_of_facilities` | 11.0% | ✅ usable (duplicate) | Matches our own `n_sites` for **all 47,210 trials**; its NaN means *0 sites* |
| `has_us_facility` | 11.0% | ✅ usable (duplicate) | Agrees with our own US flag except for 9 trials |
| `has_single_facility` | 0% | ✅ usable (duplicate) | Same information as `n_sites` = 1 |
| `minimum_age_num` / `_unit`, `maximum_age_num` / `_unit` | 5.2% / 46.8% | ✅ usable | The registered ages, split into number + unit. Units still vary (`Year`/`Years`, `Months`, `Weeks`…) and have a **leading space**, so they still need converting to years |
| `registered_in_calendar_year` | 0% | ⚠️ redundant | Same as `registration_year`, which already defines the split |
| `nlm_download_date` | 0% | ❌ not a feature | The snapshot's download date (2018-11-30), the same for every trial |
| `actual_duration` | 99.1% | ❌ leakage | Only known once a trial has ended (see the check below) |
| `were_results_reported`, `months_to_report_results` | 0% / 100% | ❌ leakage | About results, which come after the outcome (see the check below) |
| `number_of_sae_subjects`, `number_of_nsae_subjects` | 100% | ❌ leakage (and empty) | Adverse-event counts from results |

The 2018 version of this table has **no outcome-count columns**, so there's no "number of outcomes to measure" feature from here.

### Leakage check: do the "results" columns already relate to the outcome?

Even in the 2018 snapshot, a few trials already have an actual duration or reported results. If these columns are much more common among trials that later stopped (or completed), using them would leak the answer.

In [34]:
leak_check = calculated[["nct_id", "actual_duration", "were_results_reported"]].merge(
    df_trials[["nct_id", "label"]], on="nct_id")
leak_check["has_actual_duration"] = leak_check["actual_duration"].notna()

display(leak_check["were_results_reported"].value_counts(dropna=False))
# share of each group that ended up stopped early (label 1), labelled trials only
labelled = leak_check.dropna(subset=["label"])
display(labelled.groupby("has_actual_duration")["label"].agg(["count", "mean"]).round(3))
display(labelled.groupby("were_results_reported")["label"].agg(["count", "mean"]).round(3))

were_results_reported
False    47209
True         1
Name: count, dtype: int64

,count,mean
has_actual_duration,,
False,31615,0.204
True,288,0.069


,count,mean
were_results_reported,,
False,31902,0.203
True,1,0.000


### Findings – Section 4

- `calculated_values` has **16 columns** for all 47,210 cohort trials.
- **About 11% of cohort trials list no sites at all** (`number_of_facilities` is empty for them, and our own count gives 0 for exactly the same trials). That's common enough for "no sites listed yet" to be a feature in its own right.
- The pre-computed site columns (`number_of_facilities`, `has_us_facility`, `has_single_facility`) **duplicate what we built** from `facilities`, which is a good sign our counts are right. We'll use our own versions, because they follow rules we control (removed countries, NaN = 0).
- The parsed ages save the regex work, but the units still need cleaning (a leading space, singular/plural, months/weeks/days) and converting to years.
- `actual_duration`, `were_results_reported`, `months_to_report_results` and the adverse-event counts are **results information** and are excluded. The leakage check above shows whether they already relate to the label.